In [ ]:
import torch
print(torch.__version__)

### Using transformer pipeline (doesn't work)

In [ ]:
from transformers import pipeline

pipe = pipeline("image-text-to-text", model="Qwen/Qwen2-VL-2B", torch_dtype=torch.float32, device='cpu') # Qwen/Qwen3-VL-2B-Instruct
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
pipe(text=messages)

### Using AutoTokenizer

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B",
    torch_dtype=torch.float32,
)
model = model.to(device)
model.eval()

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B")

image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image_url},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]

# Step 1: extract vision inputs first
image_inputs, video_inputs = process_vision_info(messages)

# Step 2: build text — verify it's non-empty
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("TEXT TEMPLATE:", repr(text))  # should be a non-empty string
assert len(text.strip()) > 0, "apply_chat_template returned empty string!"

# Step 3: process everything together
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs if video_inputs else None,
    padding=True,
    return_tensors="pt",
)

print("input_ids shape:", inputs["input_ids"].shape)  # sanity check

# Step 4: cast + move to device
inputs = {
    k: (v.long() if k in ("input_ids", "attention_mask", "token_type_ids", "mm_token_type_ids")
        else v).to(device)
    for k, v in inputs.items()
}

with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=128)

generated_ids_trimmed = [
    out[len(inp):] for inp, out in zip(inputs["input_ids"], generated_ids)
]
print(processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True))

In [ ]:
inputs

### Using AutoModel (not working)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-1.5B") # Qwen/Qwen2-VL-2B
model = PeftModel.from_pretrained(base_model, "./outputs")

tokenizer = AutoTokenizer.from_pretrained("./outputs")

prompt = """Question:
What is the capital of Kenya?

Options:
A. Lagos
B. Nairobi
C. Cairo
D. Accra

Answer:"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=5)

print(tokenizer.decode(outputs[0]))